In [ ]:
import pandas as pd

items_df = pd.read_csv("Dataset/boq_line_items.csv")
projects_df = pd.read_csv("Dataset/boq_projects.csv")

print(items_df.columns.tolist())
print(projects_df.columns.tolist())

['item_id', 'project_id', 'category', 'sub_category', 'item_name', 'unit', 'quantity', 'rate_per_unit_inr', 'amount_inr']
['project_id', 'building_type', 'city', 'state', 'built_up_area_sqft', 'num_floors', 'project_start_date', 'estimated_duration_months', 'total_civil_cost_inr', 'total_electrical_cost_inr', 'total_plumbing_cost_inr', 'total_project_cost_inr', 'cost_per_sqft_inr']


In [3]:
merged_df = pd.merge(
    items_df,
    projects_df,
    on="project_id",
    how="inner"
)

print("Merged dataset shape:", merged_df.shape)
print(merged_df.head())

Merged dataset shape: (25644, 21)
     item_id project_id category   sub_category  \
0  ITM000001    PRJ0001    Civil       Concrete   
1  ITM000002    PRJ0001    Civil      Earthwork   
2  ITM000003    PRJ0001    Civil        Roofing   
3  ITM000004    PRJ0001    Civil  Reinforcement   
4  ITM000005    PRJ0001    Civil        Masonry   

                          item_name unit   quantity  rate_per_unit_inr  \
0      RCC M25 for footings/columns  cum    1936.67            8063.14   
1  Backfilling with excavated earth  cum    1476.05             125.48   
2  RCC roof slab with waterproofing  sqm   73651.01            1163.75   
3             TMT steel bars Fe 500   kg  415703.01              67.25   
4       Brickwork in CM 1:6 (230mm)  sqm   41611.05             668.92   

    amount_inr       building_type  ...      state built_up_area_sqft  \
0  15615641.34  Commercial - Hotel  ...  Rajasthan            86553.0   
1    185214.75  Commercial - Hotel  ...  Rajasthan            86553.

In [4]:
painting_df = merged_df[
    merged_df["sub_category"].astype(str).str.strip().str.casefold() == "painting"
].copy()

print("Painting dataset shape:", painting_df.shape)
print(painting_df.head())

Painting dataset shape: (1218, 21)
      item_id project_id category sub_category  \
8   ITM000009    PRJ0001    Civil     Painting   
40  ITM000041    PRJ0002    Civil     Painting   
41  ITM000042    PRJ0002    Civil     Painting   
65  ITM000066    PRJ0003    Civil     Painting   
68  ITM000069    PRJ0003    Civil     Painting   

                            item_name unit   quantity  rate_per_unit_inr  \
8         Exterior weatherproof paint  sqm   94984.77             114.83   
40  Interior emulsion paint (2 coats)  sqm  119559.19              98.78   
41        Exterior weatherproof paint  sqm   58969.34             114.69   
65        Exterior weatherproof paint  sqm    1686.23             109.60   
68  Interior emulsion paint (2 coats)  sqm    4842.65              93.52   

     amount_inr                     building_type  ...      state  \
8   10907101.14                Commercial - Hotel  ...  Rajasthan   
40  11810056.79  Industrial - Heavy Manufacturing  ...    Gujarat   


In [5]:
painting_df.to_csv("painting_dataset.csv", index=False)

print("Dataset saved as painting_dataset.csv")

Dataset saved as painting_dataset.csv


In [ ]:
print("Total painting samples:", len(painting_df))

print(painting_df["sub_category"].value_counts())

print(painting_df.isnull().sum())

Total painting samples: 1218
sub_category
Painting    1218
Name: count, dtype: int64
item_id                       0
project_id                    0
category                      0
sub_category                  0
item_name                     0
unit                          0
quantity                      0
rate_per_unit_inr            19
amount_inr                   19
building_type                 0
city                          0
state                         0
built_up_area_sqft            0
num_floors                    0
project_start_date            0
estimated_duration_months    27
total_civil_cost_inr          0
total_electrical_cost_inr     0
total_plumbing_cost_inr       0
total_project_cost_inr        0
cost_per_sqft_inr             0
dtype: int64


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("painting_dataset.csv")

print("Original dataset shape:", df.shape)

df = df.drop_duplicates()
df.columns = df.columns.str.strip()
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.drop_duplicates(subset=["item_id"], keep="first")

numeric_cols = [
    "quantity",
    "rate_per_unit_inr",
    "amount_inr",
    "built_up_area_sqft",
    "num_floors",
    "estimated_duration_months",
    "total_civil_cost_inr",
    "total_electrical_cost_inr",
    "total_plumbing_cost_inr",
    "total_project_cost_inr",
    "cost_per_sqft_inr"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nMissing values before handling:")
print(df.isnull().sum())

essential_cols = [
    "project_id",
    "item_name",
    "quantity",
    "unit",
    "amount_inr"
]

df = df.dropna(subset=essential_cols)

df = df[
    (df["quantity"] >= 0) &
    (df["amount_inr"] >= 0)
].copy()

text_cols = df.select_dtypes(include=["object", "str"]).columns

for col in text_cols:
    df[col] = df[col].str.strip()

print("\nMissing values after cleaning:")
print(df.isnull().sum())
print("\nFinal dataset shape:", df.shape)
print("\nDataset information:")

df.info()
df.to_csv("painting_dataset_cleaned.csv", index=False)

print("\nCleaned dataset saved as painting_dataset_cleaned.csv")

Original dataset shape: (1218, 21)

Missing values before handling:
item_id                       0
project_id                    0
category                      0
sub_category                  0
item_name                     0
unit                          0
quantity                      0
rate_per_unit_inr            19
amount_inr                   19
building_type                 0
city                          0
state                         0
built_up_area_sqft            0
num_floors                    0
project_start_date            0
estimated_duration_months    27
total_civil_cost_inr          0
total_electrical_cost_inr     0
total_plumbing_cost_inr       0
total_project_cost_inr        0
cost_per_sqft_inr             0
dtype: int64

Missing values after cleaning:
item_id                       0
project_id                    0
category                      0
sub_category                  0
item_name                     0
unit                          0
quantity               